### <b>Single Agent SQL Generation</b>
#### DuckDB-NSQL Model
DuckDB-NSQL is a 7 billion parameter text-to-SQL model designed specifically for SQL generation tasks.
This model is based on Meta’s original Llama-2 7B model and further pre-trained on a dataset of general SQL queries and then fine-tuned on a dataset composed of DuckDB text-to-SQL pairs.

#### Ollama 
Ollama is a framework designed for running large language models (LLMs) on local machine. It provides an efficient way to deploy, manage, and interact with open-source models without relying on cloud-based APIs.
Withing the scope of this research, Ollama is deployed locally to run duckdb-nsql model.

SQL data generated shall be saved in duckdb-nsql.txt and be evaluated utilizing spider evaluation benchmark.

#### Spider Benchmark
The Spider Benchmark is a widely used dataset for evaluating the text-to-SQL capabilities of machine learning models. 

Consists of:
- Database Schema: Metadata about tables, columns, and relationships.
- Natural Language Questions: User queries in English.
- Gold SQL Queries: Correct SQL output for each question.

Evaluation Metrics:
- Exact Match Accuracy – Compares generated SQL queries with the gold standard.
- Execution Accuracy – Compares execution results instead of query syntax.
- Component Match – Evaluates different SQL components (SELECT, WHERE, GROUP BY, etc.).

In [1]:

sql_expert_config = {
    "model": "duckdb-nsql:7b-q5_K_S",
    "base_url": "http://127.0.0.1:11434/v1",
    "api_key":"placeholder",
}

- DuckDB-NSQL expects the schema in the system prompt as input.
- Prompt constructed in a way to contain both the schema and natural language question.
- Model output is expected to output only the SQL query.

In [2]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelInfo, ModelFamily

user_prompt =  """ 
Provided this schema:
{table_metadata_string}


{user_question}
"""

sql_constructor_client = OpenAIChatCompletionClient(
        model=sql_expert_config["model"],
        base_url=sql_expert_config["base_url"],
        api_key=sql_expert_config["api_key"],
        model_info = ModelInfo(
            vision=False, function_calling=False,
            json_output=False, family=ModelFamily.UNKNOWN
        ),
)

/home/simges/autogen/autogen/python/packages/autogen-ext/src/autogen_ext/models/openai/_openai_client.py:412: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


- Iterate through Spider-Dev natural language questions.
- Construct user message with database schema & natural language question.
- Get model response alone and save it into duckdb-nsql.txt
- duckdb-nsql.txt shall be evaluated using Spider evaluate.py.

In [3]:
from spider_env import SpiderEnv
gym = SpiderEnv()
from autogen_core.models import UserMessage

for i in range(0, len(gym._dataset)):
    observation, info = gym.reset(k=i)
    g_question = observation["instruction"]
    g_schema = info["schema"]

    # Define the user message
    messages = [
        UserMessage(content=user_prompt.format(user_question=g_question, table_metadata_string=g_schema), source="user"),
    ]
    response = await sql_constructor_client.create(messages=messages)
    with open("duckdb-nsql.txt", "a") as file:
        final_sql = response.content.strip()
        file.write(f"{final_sql}\n")

Loading cached Spider dataset from /home/simges/.cache
Schema file not found for /home/simges/.cache/spider/database/voter_1
Schema file not found for /home/simges/.cache/spider/database/world_1


APIConnectionError: Connection error.